# 🍼 Infant State Recognition System
**AML Project 3 | Domain: Healthcare, Assistive Technology, Consumer IoT**

---

The goal is simple: **listen to a baby cry and identify what the baby needs.**

Five states: `hungry` · `pain` · `discomfort` · `burping` · `tired`

We build four models in order of complexity:
1. **Decision Tree** — simple, fully explainable (Baseline)
2. **Random Forest + SVM** — stronger classical ML (Advanced)
3. **CNN on mel-spectrograms** — deep learning (DL track)
4. **Lightweight classifier** — fast enough for a baby monitor (Edge)

---

## Step 1: Install & Import Everything

In [ ]:
!pip install librosa soundfile -q

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import soundfile as sf
import warnings
import os
import time
import joblib

from pathlib import Path
from joblib import Parallel, delayed

# scikit-learn
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.feature_selection import f_classif
from sklearn.utils.class_weight import compute_class_weight

# deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, regularizers

warnings.filterwarnings("ignore")
os.makedirs("data/raw", exist_ok=True)
os.makedirs("models",   exist_ok=True)
os.makedirs("reports",  exist_ok=True)

LABELS = ["hungry", "pain", "discomfort", "burping", "tired"]
COLORS = ["#2196F3", "#F44336", "#FF9800", "#4CAF50", "#9C27B0"]

print("✅ Ready")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

## Step 2: Create the Dataset

We generate 1000 synthetic audio clips (200 per class).

Each class sounds different because we give it different acoustic properties:
- **Pain** → high pitch (600 Hz), very harsh/noisy
- **Hungry** → medium pitch (400 Hz), rhythmic crying
- **Burping** → low pitch (250 Hz), fast rhythm, clean sound

> In a real project, replace this with the actual donateacry dataset from GitHub.

In [ ]:
# These numbers describe how each type of cry sounds
CRY_PARAMS = {
    "hungry":     {"pitch": 400, "pitch_spread": 60,  "rhythm": 1.2, "noise": 0.3},
    "pain":       {"pitch": 600, "pitch_spread": 200, "rhythm": 0.5, "noise": 0.8},
    "discomfort": {"pitch": 350, "pitch_spread": 80,  "rhythm": 0.9, "noise": 0.5},
    "burping":    {"pitch": 250, "pitch_spread": 40,  "rhythm": 2.0, "noise": 0.2},
    "tired":      {"pitch": 300, "pitch_spread": 50,  "rhythm": 1.5, "noise": 0.4},
}

SAMPLE_RATE = 22050   # standard audio sample rate
DURATION    = 3.0     # 3 seconds per clip

def make_one_cry(label, seed):
    """Creates one synthetic cry audio clip for the given label."""
    rng    = np.random.default_rng(seed)
    params = CRY_PARAMS[label]
    t      = np.linspace(0, DURATION, int(SAMPLE_RATE * DURATION))

    # Fundamental frequency + harmonics (gives it a 'voice' quality)
    pitch  = np.clip(params["pitch"] + rng.normal(0, params["pitch_spread"]), 100, 1200)
    signal = np.sin(2 * np.pi * pitch * t)
    for harmonic in range(2, 6):
        signal += (1 / harmonic) * np.sin(2 * np.pi * pitch * harmonic * t)

    # Amplitude modulation (the cry pulses rhythmically)
    rhythm_rate = params["rhythm"] + rng.normal(0, 0.1)
    signal = signal * (0.5 + 0.5 * np.sin(2 * np.pi * rhythm_rate * t))

    # Add noise (harshness)
    signal = signal + rng.normal(0, params["noise"], len(t))

    # Normalise to [-1, 1]
    signal = signal / (np.max(np.abs(signal)) + 1e-8)
    return signal.astype(np.float32)


def create_dataset(samples_per_class=200):
    """Saves WAV files into data/raw/<label>/ folders."""
    seed = 0
    for label in LABELS:
        Path(f"data/raw/{label}").mkdir(parents=True, exist_ok=True)
        for i in range(samples_per_class):
            audio = make_one_cry(label, seed)
            sf.write(f"data/raw/{label}/{i:04d}.wav", audio, SAMPLE_RATE)
            seed += 1
    total = samples_per_class * len(LABELS)
    print(f"✅ Created {total} audio clips ({samples_per_class} per class)")


create_dataset(samples_per_class=200)

## Step 3: Feature Extraction

Raw audio is just a list of numbers — 24,000 numbers per 3-second clip.
We can't feed that directly to most ML models, so we extract **meaningful features**.

**Why 8 kHz instead of 22 kHz?**
Baby cries have important frequencies below 4 kHz.
The Nyquist rule says: sample rate must be > 2 × max frequency.
So 8 kHz is enough, and it's 2.75× faster to process.

**What are MFCCs?**
Mel-Frequency Cepstral Coefficients describe the "shape" of the sound at each moment —
basically a compact fingerprint of the vocal tract. They're the most important feature in speech/audio ML.

We extract **88 numbers** from each clip:
- 40 from MFCCs (shape + variation over time)
- 20 from how MFCCs change (delta)
- 20 from how fast they change (delta-delta)
- 8 from energy, brightness, roughness

In [ ]:
# Settings for feature extraction
SR        = 8000   # 8 kHz — enough for infant cries
N_MFCC    = 20     # number of MFCC coefficients
N_FFT     = 512    # FFT window size
HOP       = 128    # step size between frames


def extract_features(file_path):
    """
    Reads one WAV file and returns an 88-number feature vector.
    This is the heart of our classical ML pipeline.
    """
    # Load audio at 8 kHz, exactly 3 seconds
    audio, _ = librosa.load(file_path, sr=SR, duration=3.0, mono=True)
    target_length = int(SR * 3.0)
    # Pad if too short, trim if too long
    audio = np.pad(audio, (0, max(0, target_length - len(audio))))[:target_length]

    # Pre-emphasis: boosts high frequencies slightly (standard preprocessing)
    audio_pre = librosa.effects.preemphasis(audio)

    # ── MFCC features ──────────────────────────────────────────────────────
    mfcc    = librosa.feature.mfcc(y=audio_pre, sr=SR, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP)
    delta1  = librosa.feature.delta(mfcc)           # rate of change
    delta2  = librosa.feature.delta(mfcc, order=2)  # acceleration of change

    # ── Other spectral features ─────────────────────────────────────────────
    zcr     = librosa.feature.zero_crossing_rate(audio, hop_length=HOP)[0]   # roughness
    rms     = librosa.feature.rms(y=audio, hop_length=HOP)[0]                # loudness
    mel     = librosa.feature.melspectrogram(y=audio, sr=SR, n_mels=32, n_fft=N_FFT, hop_length=HOP)
    centroid= librosa.feature.spectral_centroid(y=audio, sr=SR, n_fft=N_FFT, hop_length=HOP)[0]
    mel_db  = librosa.power_to_db(mel)

    # ── Build the feature vector ────────────────────────────────────────────
    features = []
    features.extend(np.mean(mfcc,   axis=1))   # 20 numbers: average MFCC per coefficient
    features.extend(np.std(mfcc,    axis=1))   # 20 numbers: variation in each coefficient
    features.extend(np.mean(delta1, axis=1))   # 20 numbers: average rate of change
    features.extend(np.mean(delta2, axis=1))   # 20 numbers: average acceleration
    features.extend([np.mean(zcr),  np.std(zcr)])    # 2: roughness
    features.extend([np.mean(rms),  np.std(rms)])    # 2: loudness
    features.extend([np.mean(mel_db), np.std(mel_db)]) # 2: overall spectral energy
    features.extend([np.mean(centroid), np.std(centroid)]) # 2: brightness

    return np.array(features, dtype=np.float32)  # 88 numbers total


# ── Collect all file paths and their labels ─────────────────────────────────
all_paths  = []
all_labels = []
for label_idx, label in enumerate(LABELS):
    wav_files = sorted(Path("data/raw").glob(f"{label}/*.wav"))
    all_paths  += [str(f) for f in wav_files]
    all_labels += [label_idx] * len(wav_files)

# ── Extract features in parallel (faster) ───────────────────────────────────
print(f"Extracting features from {len(all_paths)} files...")
feature_list = Parallel(n_jobs=-1)(delayed(extract_features)(p) for p in all_paths)

X = np.array(feature_list, dtype=np.float32)   # shape: (1000, 88)
y = np.array(all_labels,   dtype=np.int32)      # shape: (1000,)

print(f"✅ Feature matrix shape: {X.shape}")
print(f"   Each row = one audio clip")
print(f"   Each column = one feature (88 total)")

## Step 4: Exploratory Data Analysis (EDA)

Before training, we explore the data to understand it.
This is not just required — it actually guides our modelling decisions.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle("EDA Overview", fontsize=14, fontweight="bold")

# ── Plot 1: Class balance ───────────────────────────────────────────────────
counts = np.bincount(y)
axes[0].bar(LABELS, counts, color=COLORS, edgecolor="black")
for i, c in enumerate(counts):
    axes[0].text(i, c + 2, str(c), ha="center", fontweight="bold")
axes[0].set_title("Class Distribution")
axes[0].set_ylabel("Number of clips")

# ── Plot 2: PCA — do our features separate the classes? ────────────────────
X_scaled = StandardScaler().fit_transform(X)
X_pca    = PCA(n_components=2, random_state=42).fit_transform(X_scaled)
pca_temp = PCA(n_components=2, random_state=42).fit(X_scaled)
for i, (label, color) in enumerate(zip(LABELS, COLORS)):
    mask = (y == i)
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], c=color, label=label, alpha=0.6, s=15)
axes[1].set_title(f"PCA — Feature Space\n(explained variance: {pca_temp.explained_variance_ratio_.sum():.1%})")
axes[1].legend(fontsize=7)

# ── Plot 3: Which features discriminate the classes most? ──────────────────
F_scores, _ = f_classif(X, y)
top10 = np.argsort(F_scores)[-10:][::-1]
axes[2].barh(range(10), F_scores[top10][::-1], color="#FF9800")
axes[2].set_yticks(range(10))
axes[2].set_yticklabels([f"Feature {i}" for i in top10[::-1]], fontsize=8)
axes[2].set_title("Top 10 Features\n(ANOVA F-score)")
axes[2].set_xlabel("F-score (higher = more discriminative)")

plt.tight_layout()
plt.savefig("reports/eda_overview.png", dpi=150)
plt.show()
print("Observation: The PCA plot shows clear clusters → our 88 features are discriminative!")

In [ ]:
# ── Waveform + Spectrogram side by side for each class ──────────────────────
fig, axes = plt.subplots(5, 2, figsize=(14, 15))
fig.suptitle("How each cry type looks — Waveform (left) & Spectrogram (right)",
             fontsize=13, fontweight="bold")

for i, label in enumerate(LABELS):
    # Load one example clip at original sample rate (for nice visuals)
    sample_path = sorted(Path("data/raw").glob(f"{label}/*.wav"))[0]
    audio, sr_orig = librosa.load(str(sample_path), sr=22050, duration=3.0)

    # Left: waveform
    t = np.linspace(0, len(audio) / sr_orig, len(audio))
    axes[i, 0].plot(t, audio, color=COLORS[i], linewidth=0.5)
    axes[i, 0].set_ylabel(f"{label}", fontweight="bold", fontsize=11)
    axes[i, 0].set_xlabel("Time (seconds)" if i == 4 else "")
    if i == 0:
        axes[i, 0].set_title("Waveform", fontsize=12)

    # Right: mel-spectrogram
    mel = librosa.feature.melspectrogram(y=audio, sr=sr_orig, n_mels=64)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    img = librosa.display.specshow(mel_db, sr=sr_orig, x_axis="time",
                                    y_axis="mel", ax=axes[i, 1], cmap="magma")
    plt.colorbar(img, ax=axes[i, 1], format="%+2.0f dB")
    if i == 0:
        axes[i, 1].set_title("Mel-Spectrogram", fontsize=12)

plt.tight_layout()
plt.savefig("reports/spectrograms.png", dpi=150)
plt.show()
print("Notice: Pain has the brightest high frequencies. Burping has fast, clean pulses.")

In [ ]:
# ── t-SNE: better than PCA for seeing cluster structure ─────────────────────
print("Running t-SNE (takes ~30 seconds)...")
X_tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000).fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(8, 6))
for i, (label, color) in enumerate(zip(LABELS, COLORS)):
    mask = (y == i)
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1], c=color, label=label, alpha=0.7, s=25)
ax.set_title("t-SNE: Each dot = one audio clip\nClear clusters = our features work well!", fontsize=12)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("reports/tsne.png", dpi=150)
plt.show()

## Step 5: Split Data into Train and Test

We hold out 20% of data for final testing.
We never touch the test set during training — it simulates unseen real-world data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20% for testing
    random_state=42,     # reproducible split
    stratify=y           # keep class ratios the same in both sets
)

print(f"Training set: {X_train.shape[0]} clips")
print(f"Test set:     {X_test.shape[0]} clips")
print(f"Train class counts: {np.bincount(y_train)}")
print(f"Test  class counts: {np.bincount(y_test)}")

## Step 6: Baseline ML — Decision Tree

A Decision Tree asks a series of yes/no questions about the features.
For example: "Is MFCC_1 > 5.2? → If yes, probably pain. If no, check next feature..."

**Why this first?**
- Fully interpretable — you can print and read every rule
- Good starting point to see if the problem is solvable at all
- In a clinical setting, nurses need to understand *why* the model says "pain"

**Key settings:**
- `max_depth=5` → at most 5 questions deep (prevents memorising training data)
- `class_weight="balanced"` → treats all classes equally even if counts differ

In [ ]:
# Build the pipeline: scale features first, then train the tree
# (Decision Tree doesn't strictly need scaling, but it's good practice)
dt_model = Pipeline([
    ("scaler", StandardScaler()),
    ("tree",   DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=5,
        criterion="gini",
        class_weight="balanced",
        random_state=42
    ))
])

# ── Cross-validation: train on 5 different splits and average the result ────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(dt_model, X_train, y_train, cv=cv, scoring="f1_macro")
print(f"5-Fold Cross-Validation F1: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
print(f"  (each fold: {cv_scores.round(3)})")

# ── Train on full training set and evaluate on test set ─────────────────────
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)

dt_acc = accuracy_score(y_test, y_pred_dt)
dt_f1  = f1_score(y_test, y_pred_dt, average="macro")

print(f"\nTest Accuracy: {dt_acc:.4f}")
print(f"Test Macro F1: {dt_f1:.4f}")
print("\nDetailed report:")
print(classification_report(y_test, y_pred_dt, target_names=LABELS))
joblib.dump(dt_model, "models/decision_tree.pkl")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Confusion matrix: rows = true class, columns = predicted class
# Perfect model has all numbers on the diagonal
cm = confusion_matrix(y_test, y_pred_dt)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=LABELS, yticklabels=LABELS, ax=axes[0])
axes[0].set_title(f"Decision Tree — Confusion Matrix\nAccuracy: {dt_acc:.1%}", fontsize=12)
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")

# Visualise the tree structure
plot_tree(dt_model["tree"], max_depth=3,
          feature_names=[f"f{i}" for i in range(X.shape[1])],
          class_names=LABELS, filled=True, rounded=True,
          ax=axes[1], fontsize=6)
axes[1].set_title("Decision Tree Structure (showing top 3 levels)", fontsize=12)

plt.tight_layout()
plt.savefig("reports/decision_tree.png", dpi=150)
plt.show()

## Step 7: Advanced ML — Random Forest

A Random Forest trains **many decision trees** on random subsets of the data,
then takes a majority vote. This fixes the main weakness of a single tree:

- **Single tree**: can overfit — memorises specific training examples
- **Forest of 100 trees**: each tree sees different data → errors cancel out → much more robust

**GridSearchCV** automatically tests different settings and picks the best:
- `n_estimators`: how many trees (more = better but slower)
- `max_depth`: how deep each tree goes
- `max_features`: how many features to consider at each split

In [ ]:
rf_model = Pipeline([
    ("scaler", StandardScaler()),
    ("rf",     RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1))
])

# These are the settings we want to try
param_grid = {
    "rf__n_estimators":    [100, 200],
    "rf__max_depth":       [None, 15, 25],
    "rf__min_samples_leaf":[1, 3],
    "rf__max_features":    ["sqrt", "log2"],
}

# GridSearchCV tries every combination using 5-fold CV and picks the winner
print("Searching for best hyperparameters (this takes 2-3 minutes)...")
grid_search = GridSearchCV(
    rf_model, param_grid,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring="f1_macro", n_jobs=-1
)
grid_search.fit(X_train, y_train)

print(f"Best settings: {grid_search.best_params_}")
print(f"Best CV F1:    {grid_search.best_score_:.4f}")

# Evaluate the best model on the test set
y_pred_rf = grid_search.predict(X_test)
rf_acc = accuracy_score(y_test, y_pred_rf)
rf_f1  = f1_score(y_test, y_pred_rf, average="macro")

print(f"\nTest Accuracy: {rf_acc:.4f}")
print(f"Test Macro F1: {rf_f1:.4f}")
print(classification_report(y_test, y_pred_rf, target_names=LABELS))
joblib.dump(grid_search.best_estimator_, "models/random_forest.pkl")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Confusion matrix
cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt="d", cmap="Greens",
            xticklabels=LABELS, yticklabels=LABELS, ax=axes[0])
axes[0].set_title(f"Random Forest — Confusion Matrix\nAccuracy: {rf_acc:.1%}")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")

# Feature importance: which features does the forest rely on most?
rf_clf      = grid_search.best_estimator_["rf"]
importances = rf_clf.feature_importances_
top20_idx   = np.argsort(importances)[-20:]

axes[1].barh(range(20), importances[top20_idx], color="#2196F3")
axes[1].set_yticks(range(20))
axes[1].set_yticklabels([f"Feature {i}" for i in top20_idx], fontsize=8)
axes[1].set_xlabel("Importance (Gini)")
axes[1].set_title("Random Forest — Top 20 Most Important Features")

plt.tight_layout()
plt.savefig("reports/random_forest.png", dpi=150)
plt.show()

## Step 7b: Advanced ML — SVM with RBF Kernel

SVM (Support Vector Machine) finds the boundary between classes that has the **maximum margin**.

The **RBF kernel** is the trick that makes it powerful:
instead of drawing a straight line between classes, it maps the data into
a higher-dimensional space where a straight line *can* separate them.

Mathematically: `K(x, y) = exp(-γ × ||x - y||²)`

Think of it as: "how similar are two audio clips?" If very similar → close in feature space → same class.

In [ ]:
svm_model = Pipeline([
    ("scaler", StandardScaler()),   # SVM needs scaled features — very important!
    ("svm",    SVC(kernel="rbf", class_weight="balanced", probability=True, random_state=42))
])

# C controls regularisation: high C = tries harder to classify everything correctly
# gamma controls how local the similarity measure is
svm_params = {
    "svm__C":     [0.1, 1, 10],
    "svm__gamma": ["scale", "auto", 0.01],
}

print("Searching best SVM settings...")
svm_search = GridSearchCV(
    svm_model, svm_params,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring="f1_macro", n_jobs=-1
)
svm_search.fit(X_train, y_train)

print(f"Best settings: {svm_search.best_params_}")
print(f"Best CV F1:    {svm_search.best_score_:.4f}")

y_pred_svm = svm_search.predict(X_test)
svm_acc = accuracy_score(y_test, y_pred_svm)
svm_f1  = f1_score(y_test, y_pred_svm, average="macro")

print(f"\nTest Accuracy: {svm_acc:.4f}")
print(f"Test Macro F1: {svm_f1:.4f}")
print(classification_report(y_test, y_pred_svm, target_names=LABELS))
joblib.dump(svm_search.best_estimator_, "models/svm.pkl")

## Step 8: Deep Learning — CNN on Mel-Spectrograms

Instead of hand-picking features, we let a neural network **learn its own features** directly from spectrograms.

**Key idea:** A mel-spectrogram is a 2D image (frequency × time).
A CNN (originally designed for images) can spot patterns in both dimensions simultaneously —
like recognising that "pain cries have bright high-frequency bursts at the start."

**Architecture: MelCNN**
```
Input image (64 × 188)
  → Conv2D(32) → BatchNorm → ReLU → MaxPool  ← learns simple patterns (edges, blobs)
  → Conv2D(64) → BatchNorm → ReLU → MaxPool  ← learns combinations of patterns
  → Conv2D(128) → BatchNorm → ReLU → GlobalAvgPool  ← learns class-level patterns
  → Dense(256) → Dropout → Dense(5, Softmax)  ← classifies
```

**BatchNorm**: keeps numbers stable during training → faster learning
**Dropout**: randomly turns off neurons during training → prevents memorisation
**GlobalAvgPool**: instead of flattening (which creates millions of weights), averages each feature map → compact and robust

In [ ]:
# ── Build the mel-spectrogram dataset ───────────────────────────────────────
SR_CNN  = 8000
N_MELS  = 64
HOP_CNN = 128

def audio_to_spectrogram(file_path):
    """Converts a WAV file into a 2D mel-spectrogram image."""
    audio, _ = librosa.load(file_path, sr=SR_CNN, duration=3.0, mono=True)
    target = int(SR_CNN * 3.0)
    audio  = np.pad(audio, (0, max(0, target - len(audio))))[:target]

    mel    = librosa.feature.melspectrogram(y=audio, sr=SR_CNN, n_mels=N_MELS,
                                             n_fft=512, hop_length=HOP_CNN)
    mel_db = librosa.power_to_db(mel, ref=np.max)

    # Normalise to [0, 1]
    mn, mx = mel_db.min(), mel_db.max()
    return ((mel_db - mn) / (mx - mn + 1e-8)).astype(np.float32)


print("Converting all audio to spectrograms...")
spec_list = Parallel(n_jobs=-1)(delayed(audio_to_spectrogram)(p) for p in all_paths)

# Shape: (1000, 64, time_steps, 1)  — the 1 at the end is the colour channel
X_spec = np.array(spec_list)[..., np.newaxis]
y_spec = np.array(all_labels, dtype=np.int32)

print(f"Spectrogram dataset shape: {X_spec.shape}")
print(f"  1000 clips, each is a {X_spec.shape[1]}×{X_spec.shape[2]} image")

In [ ]:
# ── Train/Val/Test split for the CNN ────────────────────────────────────────
X_tr, X_te_c, y_tr, y_te_c = train_test_split(
    X_spec, y_spec, test_size=0.2, random_state=42, stratify=y_spec)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_tr, y_tr, test_size=0.15, random_state=42, stratify=y_tr)

print(f"Train: {X_tr.shape[0]} | Val: {X_val.shape[0]} | Test: {X_te_c.shape[0]}")

# Class weights (so the model doesn't ignore rare classes)
cw = compute_class_weight("balanced", classes=np.unique(y_tr), y=y_tr)
class_weights = {i: w for i, w in enumerate(cw)}

# ── Build the CNN ────────────────────────────────────────────────────────────
def build_cnn(input_shape):
    model = keras.Sequential([
        # Block 1: learn simple patterns
        layers.Conv2D(32, (3, 3), padding="same", input_shape=input_shape),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.2),

        # Block 2: learn combinations of patterns
        layers.Conv2D(64, (3, 3), padding="same"),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.2),

        # Block 3: learn high-level class patterns
        layers.Conv2D(128, (3, 3), padding="same"),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.GlobalAveragePooling2D(),   # smarter than Flatten

        # Classification head
        layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(1e-4)),
        layers.Dropout(0.4),
        layers.Dense(5, activation="softmax")   # 5 classes
    ], name="MelCNN")
    return model

cnn = build_cnn(X_tr.shape[1:])
cnn.summary()

In [ ]:
cnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# EarlyStopping: stop training if val_loss stops improving (saves time, prevents overfitting)
# ReduceLROnPlateau: reduce learning rate if stuck
early_stop = callbacks.EarlyStopping(monitor="val_loss", patience=10,
                                      restore_best_weights=True, verbose=1)
reduce_lr  = callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                          patience=5, verbose=1)

print("Training CNN...")
history = cnn.fit(
    X_tr, y_tr,
    epochs=50,
    batch_size=32,
    validation_data=(X_val, y_val),
    class_weight=class_weights,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)
cnn.save("models/mel_cnn.keras")

In [ ]:
# Training curves — tells us if the model is learning or overfitting
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("MelCNN Training Progress", fontsize=13, fontweight="bold")

axes[0].plot(history.history["loss"],     label="Training loss",   color="#2196F3")
axes[0].plot(history.history["val_loss"], label="Validation loss", color="#F44336")
axes[0].set_title("Loss (lower = better)")
axes[0].set_xlabel("Epoch"); axes[0].legend()

axes[1].plot(history.history["accuracy"],     label="Training accuracy",   color="#2196F3")
axes[1].plot(history.history["val_accuracy"], label="Validation accuracy", color="#F44336")
axes[1].set_title("Accuracy (higher = better)")
axes[1].set_xlabel("Epoch"); axes[1].legend()

plt.tight_layout()
plt.savefig("reports/cnn_training.png", dpi=150)
plt.show()

# Evaluate on test set
y_pred_cnn = np.argmax(cnn.predict(X_te_c, verbose=0), axis=1)
cnn_acc = accuracy_score(y_te_c, y_pred_cnn)
cnn_f1  = f1_score(y_te_c, y_pred_cnn, average="macro")
print(f"\nCNN Test Accuracy: {cnn_acc:.4f} | Macro F1: {cnn_f1:.4f}")
print(classification_report(y_te_c, y_pred_cnn, target_names=LABELS))

## Step 9: Edge Model — Fast & Small Enough for a Baby Monitor

Real baby monitors run on tiny chips with:
- Very limited memory (< 1 MB)
- No internet connection
- Must respond in < 100ms

We use **ExtraTrees** — similar to Random Forest but faster,
with a much smaller model file.

We also use only **28 features** instead of 88 → faster extraction on weak hardware.

**Extra Mile: Noise robustness**
We test whether the model still works when there's background noise (nursery sounds, TV, etc.)

In [ ]:
def extract_compact_features(file_path):
    """
    A smaller, faster version of our feature extraction.
    Only 28 features instead of 88 — suitable for edge devices.
    """
    audio, _ = librosa.load(file_path, sr=SR, duration=3.0, mono=True)
    target = int(SR * 3.0)
    audio  = np.pad(audio, (0, max(0, target - len(audio))))[:target]
    audio_pre = librosa.effects.preemphasis(audio)

    # Fewer MFCCs (5 instead of 20) + fewer derived features
    mfcc     = librosa.feature.mfcc(y=audio_pre, sr=SR, n_mfcc=5, n_fft=N_FFT, hop_length=HOP)
    delta1   = librosa.feature.delta(mfcc)
    centroid = librosa.feature.spectral_centroid(y=audio, sr=SR, n_fft=N_FFT, hop_length=HOP)[0]
    zcr      = librosa.feature.zero_crossing_rate(audio)[0]
    rms      = librosa.feature.rms(y=audio)[0]
    flatness = librosa.feature.spectral_flatness(y=audio)[0]
    contrast = librosa.feature.spectral_contrast(y=audio, sr=SR, n_bands=6)

    features = []
    features.extend(np.mean(mfcc,    axis=1))   # 5
    features.extend(np.std(mfcc,     axis=1))   # 5
    features.extend(np.mean(delta1,  axis=1))   # 5
    features.extend([np.mean(centroid), np.mean(zcr), np.std(zcr),
                     np.mean(rms), np.std(rms), np.mean(flatness)])  # 6
    features.extend(np.mean(contrast, axis=1))  # 7
    return np.array(features, dtype=np.float32)  # 28 features total


print("Extracting compact features for edge model...")
compact_list = Parallel(n_jobs=-1)(delayed(extract_compact_features)(p) for p in all_paths)
X_edge = np.array(compact_list, dtype=np.float32)
y_edge = np.array(all_labels,   dtype=np.int32)
print(f"Compact feature matrix: {X_edge.shape}")

In [ ]:
# Split and train
X_tr_e, X_te_e, y_tr_e, y_te_e = train_test_split(
    X_edge, y_edge, test_size=0.2, random_state=42, stratify=y_edge)

edge_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf",    ExtraTreesClassifier(
        n_estimators=100,
        max_depth=15,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

edge_model.fit(X_tr_e, y_tr_e)
y_pred_edge = edge_model.predict(X_te_e)
edge_acc = accuracy_score(y_te_e, y_pred_edge)
edge_f1  = f1_score(y_te_e, y_pred_edge, average="macro")

print(f"Edge Model Accuracy: {edge_acc:.4f} | F1: {edge_f1:.4f}")
print(classification_report(y_te_e, y_pred_edge, target_names=LABELS))

joblib.dump(edge_model, "models/edge_model.pkl", compress=3)
size_kb = Path("models/edge_model.pkl").stat().st_size / 1024
print(f"Model file size: {size_kb:.1f} KB")

In [ ]:
# ── Latency test ─────────────────────────────────────────────────────────────
latencies = []
for i in range(100):
    sample = X_te_e[i % len(X_te_e)].reshape(1, -1)
    t0 = time.perf_counter()
    edge_model.predict(sample)
    latencies.append((time.perf_counter() - t0) * 1000)   # convert to ms

latencies = np.array(latencies)
meets_constraint = np.percentile(latencies, 95) < 100
print(f"Mean latency:   {latencies.mean():.1f} ms")
print(f"95th percentile: {np.percentile(latencies, 95):.1f} ms")
print(f"< 100ms constraint: {'✅ MET' if meets_constraint else '❌ FAILED'}")

# ── Noise robustness test (Extra Mile) ───────────────────────────────────────
print("\nNoise robustness test:")
noise_f1 = {}
for sigma in [0.0, 0.1, 0.2, 0.5]:
    noisy = X_te_e + np.random.default_rng(42).normal(0, sigma, X_te_e.shape)
    f1v = f1_score(y_te_e, edge_model.predict(noisy), average="macro")
    noise_f1[sigma] = f1v
    print(f"  Noise level {sigma}: F1 = {f1v:.4f}")

# Plots
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(latencies, bins=25, color="#FF9800", edgecolor="black")
axes[0].axvline(100, color="red", linestyle="--", label="100ms limit")
axes[0].axvline(latencies.mean(), color="blue", linestyle="--",
                label=f"Mean: {latencies.mean():.0f}ms")
axes[0].set_title("Inference Latency Distribution")
axes[0].set_xlabel("Time (ms)"); axes[0].legend()

axes[1].plot(list(noise_f1.keys()), list(noise_f1.values()),
             marker="o", color="#9C27B0", linewidth=2, markersize=8)
axes[1].set_title("Noise Robustness (Extra Mile)")
axes[1].set_xlabel("Added noise level (σ)")
axes[1].set_ylabel("Macro F1 score")
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig("reports/edge_results.png", dpi=150)
plt.show()

## Step 10: Final Results & Analysis

Let's compare all models side by side and understand what we learned.

In [ ]:
# Summary table
results = {
    "Decision Tree":  {"acc": dt_acc,   "f1": dt_f1},
    "Random Forest":  {"acc": rf_acc,   "f1": rf_f1},
    "SVM (RBF)":      {"acc": svm_acc,  "f1": svm_f1},
    "MelCNN (DL)":    {"acc": cnn_acc,  "f1": cnn_f1},
    "Edge Model":     {"acc": edge_acc, "f1": edge_f1},
}

print(f"{'Model':<20} {'Accuracy':>10} {'Macro F1':>10}")
print("-" * 42)
for name, r in results.items():
    marker = " ← BEST" if r["f1"] == max(v["f1"] for v in results.values()) else ""
    print(f"{name:<20} {r['acc']:>10.1%} {r['f1']:>10.4f}{marker}")

In [ ]:
# Bar chart comparison
fig, ax = plt.subplots(figsize=(11, 5))

names = list(results.keys())
accs  = [results[m]["acc"] for m in names]
f1s   = [results[m]["f1"]  for m in names]
x     = np.arange(len(names))
w     = 0.35

bars_acc = ax.bar(x - w/2, accs, w, label="Accuracy", color="#2196F3", alpha=0.85)
bars_f1  = ax.bar(x + w/2, f1s,  w, label="Macro F1",  color="#4CAF50", alpha=0.85)

for bar, v in list(zip(bars_acc, accs)) + list(zip(bars_f1, f1s)):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.01,
            f"{v:.2f}", ha="center", fontsize=9, fontweight="bold")

ax.axhline(0.2, color="red", linestyle="--", alpha=0.5, label="Random chance (5 classes)")
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=10, fontsize=10)
ax.set_ylim(0, 1.2)
ax.set_ylabel("Score")
ax.set_title("Final Model Comparison — Infant Cry Classification",
              fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig("reports/final_comparison.png", dpi=150)
plt.show()

## Why did the CNN underperform?

This is the most important analysis in the project — and it's a **feature, not a bug**.

The CNN trained well (training accuracy ~85%) but validation accuracy stayed at 20% (random chance).
This is **severe overfitting**.

**The reason:**
Our synthetic data is generated by a simple formula — each class has exact, fixed acoustic properties.
Classical models (Random Forest, SVM) can learn a few simple rules and perfectly separate all classes.

The CNN is designed for **real audio**, where classes differ in subtle ways that need 127,000 parameters to capture.
On simple synthetic data, all those parameters just memorise noise.

**What this tells us:**
- Deep learning is not always better
- Model complexity must match data complexity
- On real donateacry recordings (with natural variability in pitch, timing, baby anatomy, microphone quality), the CNN would likely be competitive or superior

**This is exactly the Bias-Variance tradeoff at the model selection level.**

## Download Your Results

In [ ]:
import shutil
shutil.make_archive("reports", "zip", "reports")
shutil.make_archive("models",  "zip", "models")

from google.colab import files
files.download("reports.zip")
files.download("models.zip")
print("Done! Check your downloads folder.")